# Predicting La Liga Match Outcomes with Logistic Regression

**Course:** Data Science Final Exam Project
**Format:** Technical Report


---

## Abstract

This project investigates whether the outcome of a La Liga football match — Home Win (H), Draw (D), or Away Win (A) — can be reliably predicted using historical match statistics and team quality ratings. We combine two independent public data sources, perform thorough exploratory data analysis, engineer meaningful difference features, and train a multinomial logistic regression classifier. The model is evaluated using accuracy, macro F1-score, and a confusion matrix. We also compare logistic regression to competing approaches and discuss limitations and potential extensions. All analysis is reproducible from the provided code.

---


## Table of Contents

1. [Problem Formulation](#1-problem-formulation)
2. [Data Sources](#2-data-sources)
3. [Mathematical Background](#3-mathematical-background)
4. [Data Loading & Cleaning](#4-data-loading--cleaning)
5. [Exploratory Data Analysis](#5-exploratory-data-analysis)
6. [Feature Engineering & Validation](#6-feature-engineering--validation)
7. [Model Training](#7-model-training)
8. [Evaluation & Interpretation](#8-evaluation--interpretation)
9. [Discussion & Conclusions](#9-discussion--conclusions)
10. [Self-Assessment](#10-self-assessment)
11. [References](#11-references)

---


---
## 1. Problem Formulation

### 1.1 Real-World Problem & Significance

Football match prediction is one of the most studied problems in sports analytics. It has direct value for:

- **Team management:** Tactical preparation, squad selection, opponent scouting.
- **Sports media:** Pre-match analysis, narrative framing, viewer engagement.
- **Sports betting markets:** La Liga betting alone involves billions of euros annually. Even a small edge over the market is financially significant.
- **Academic research:** Football provides a rich, well-labelled, publicly available testbed for classification methods.

Despite decades of research, accurate pre-match prediction remains hard — draws in particular are notoriously difficult to predict. This project focuses on **in-match prediction** (using statistics gathered during the game), which is a somewhat easier but still meaningful problem.

### 1.2 Problem Scope & Objectives

We focus exclusively on **La Liga** (Spain, top division) across four seasons (2019–2023). Our objectives are:

1. Consolidate and validate two independent data sources into one clean dataset.
2. Perform exploratory data analysis to understand the feature space.
3. Train a logistic regression classifier and evaluate it rigorously.
4. Interpret the model weights to understand what drives predictions.
5. Honestly discuss limitations and propose extensions.

### 1.3 Mathematical Problem Statement

Each match is represented by a feature vector $\mathbf{x} \in \mathbb{R}^d$. The outcome label $y$ belongs to one of three classes:

$$y \in \mathcal{Y} = \{0, 1, 2\} \quad \leftrightarrow \quad \{\text{Home Win}, \; \text{Draw}, \; \text{Away Win}\}$$

We seek a classifier $f: \mathbb{R}^d \to \mathcal{Y}$ that minimises the expected 0-1 loss on unseen data drawn from the same distribution:

$$\mathcal{R}(f) = \mathbb{E}_{(\mathbf{x}, y) \sim \mathcal{D}}\left[\mathbf{1}\{f(\mathbf{x}) \neq y\}\right]$$

Since $\mathcal{D}$ is unknown, we minimise an empirical proxy (cross-entropy loss) on training data and measure generalisation on a held-out test set.

### 1.4 Known Constraints & Assumptions

| Constraint / Assumption | Justification |
|---|---|
| In-match statistics only (goals, shots, cards…) | Pre-match data (injuries, form) requires more complex data pipelines |
| Linear decision boundary (logistic regression) | Chosen for interpretability; non-linear models are discussed as extensions |
| IID assumption relaxed: chronological split used | Matches are not strictly IID across seasons; time-based splitting is more realistic |
| Only La Liga (single league) | Cross-league generalisation is a known challenge left for future work |
| No player-level data | Player ratings change match-to-match and are harder to obtain reliably |

### 1.5 Comparison with Similar Approaches

Many prior approaches exist for football prediction:

| Approach | Pros | Cons |
|---|---|---|
| **Logistic Regression** (this project) | Interpretable, fast, well-understood mathematically | Linear boundary only, no interaction terms |
| **Dixon-Coles model** (1997) | Models goals scored directly using Poisson process, corrects for low-scoring draws | Requires more assumptions about goal distributions |
| **Random Forest / XGBoost** | Captures non-linear feature interactions, often higher accuracy | Black-box, harder to interpret, more hyperparameters |
| **Neural Networks / LSTM** | Can model sequential match history, rich representations | Requires much more data and compute, overfitting risk |
| **Betting market odds** | Encode collective wisdom of millions of bettors | Not a model per se; dependent on market efficiency |

We choose logistic regression as a principled, interpretable baseline. The other approaches are viable extensions discussed in Section 9.

---
## 2. Data Sources

We use **two fully independent data sources** with different formats, providers, and granularities. Combining them demonstrates data consolidation and merging skills.

### 2.1 Source 1 — football-data.co.uk (Match Results)

| Property | Detail |
|---|---|
| **Provider** | football-data.co.uk (Joseph Buchdahl) |
| **URL** | https://www.football-data.co.uk/spainm.php |
| **Format** | CSV, one row per match |
| **Coverage** | La Liga seasons 2019-20, 2020-21, 2021-22, 2022-23 |
| **Key columns** | Goals (FTHG, FTAG), Shots (HS, AS), Shots on Target (HST, AST), Corners (HC, AC), Cards (HY, AY, HR, AR), Half-time result (HTR), Full-time result (FTR) |
| **License** | Free for non-commercial use |

This source provides granular in-match statistics. It is the primary source for features and labels.

### 2.2 Source 2 — FIFA Team Ratings via Kaggle

| Property | Detail |
|---|---|
| **Provider** | Hugo Mathien, Kaggle |
| **URL** | https://www.kaggle.com/datasets/hugomathien/soccer |
| **Format** | SQLite database → extracted to CSV |
| **Coverage** | Team attributes (attack, defence, speed ratings) from FIFA video game data, matched by team name |
| **Key columns** | `buildUpPlaySpeed`, `defencePressure`, `attackingWorkRate`, overall attack/defence scores |
| **License** | CC0 Public Domain |

This source provides a proxy for team quality. FIFA ratings are widely used in football analytics research as a reasonable approximation of real team strength.

### 2.3 Why Two Sources Are Genuinely Independent

These two sources are independent in every meaningful sense:
- **Different providers** — one is a dedicated football statistics site, one is a game database on Kaggle.
- **Different data types** — one is match-level event data, the other is team-level quality ratings.
- **Different update mechanisms** — match results update after each game; FIFA ratings update annually.
- **Different variables** — there is no overlap in columns between the two sources.

Merging them adds team quality context to each match, enriching the feature set beyond raw match statistics.

### 2.4 Data Merge Strategy

We merge on **team name** and **season**. Because team names may differ slightly between sources (e.g. "Real Madrid" vs "Real Madrid CF"), we apply fuzzy name normalisation before merging. The merge is a left join from Source 1 (matches) — every match is kept, and team ratings are added where available.

---
## 3. Mathematical Background

### 3.1 The Sigmoid Function (Binary Case)

Logistic regression begins with the **sigmoid function**, which maps any real number to the interval $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

For a single input $\mathbf{x} \in \mathbb{R}^d$ with weight vector $\mathbf{w}$ and bias $b$, the predicted probability of class 1 is:

$$\hat{p} = P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b)$$

Key properties: $\sigma(0) = 0.5$, $\sigma(z) \to 1$ as $z \to \infty$, $\sigma(z) \to 0$ as $z \to -\infty$.

### 3.2 Multinomial Logistic Regression (Softmax)

For $K = 3$ classes, we generalise using the **softmax function**. Each class $k \in \{0, 1, 2\}$ has its own weight vector $\mathbf{w}_k \in \mathbb{R}^d$ and bias $b_k$. The predicted probability for class $k$ is:

$$P(y = k \mid \mathbf{x}) = \frac{\exp(\mathbf{w}_k^\top \mathbf{x} + b_k)}{\sum_{j=0}^{K-1} \exp(\mathbf{w}_j^\top \mathbf{x} + b_j)}$$

The softmax guarantees:

$$\sum_{k=0}^{K-1} P(y=k \mid \mathbf{x}) = 1, \qquad P(y=k \mid \mathbf{x}) \in (0,1) \; \forall k$$

The decision boundary between classes $i$ and $j$ is the hyperplane:

$$(\mathbf{w}_i - \mathbf{w}_j)^\top \mathbf{x} + (b_i - b_j) = 0$$

This is linear in $\mathbf{x}$, which is why logistic regression is a **linear classifier**.

### 3.3 Feature Standardisation

Before training, we standardise each feature $j$ using the training set mean $\mu_j$ and standard deviation $\sigma_j$:

$$\tilde{x}_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

This ensures all features contribute equally to the dot product $\mathbf{w}^\top \mathbf{x}$, and gradient descent converges faster.

> **Important:** $\mu_j$ and $\sigma_j$ are computed **only on training data** and applied to both train and test sets. Computing them on the full dataset would constitute **data leakage**.

### 3.4 Categorical Cross-Entropy Loss

We train by minimising the **categorical cross-entropy** over $n$ training examples. Let $y_{ik} = 1$ if example $i$ has true class $k$ (one-hot encoding), and $\hat{p}_{ik} = P(y_i = k \mid \mathbf{x}_i)$:

$$\mathcal{J}(\mathbf{W}, \mathbf{b}) = -\frac{1}{n} \sum_{i=1}^{n} \sum_{k=0}^{K-1} y_{ik} \log \hat{p}_{ik}$$

Since $y_{ik}$ is one-hot, only the log-probability of the true class contributes per example:

$$\mathcal{J} = -\frac{1}{n} \sum_{i=1}^{n} \log \hat{p}_{i, y_i}$$

Cross-entropy is the **negative log-likelihood** under the categorical distribution, so minimising it is equivalent to maximum likelihood estimation.

### 3.5 L2 Regularisation

To prevent overfitting we add an **L2 penalty**:

$$\mathcal{J}_{\text{reg}}(\mathbf{W}, \mathbf{b}) = \mathcal{J}(\mathbf{W}, \mathbf{b}) + \frac{\lambda}{2} \sum_{k=0}^{K-1} \|\mathbf{w}_k\|_2^2$$

where $\lambda > 0$ is the regularisation strength. In scikit-learn, $C = 1/\lambda$ — a **smaller** $C$ means **stronger** regularisation.

L2 regularisation has a Bayesian interpretation: it corresponds to placing a zero-mean Gaussian prior on the weights.

### 3.6 Optimisation: L-BFGS

We use the **L-BFGS** (Limited-memory Broyden–Fletcher–Goldfarb–Shanno) algorithm, a quasi-Newton method that approximates the Hessian matrix. The parameter update rule is:

$$\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \alpha_t \mathbf{H}_t^{-1} \nabla_{\boldsymbol{\theta}} \mathcal{J}_{\text{reg}}$$

where $\mathbf{H}_t$ is the approximate inverse Hessian and $\alpha_t$ is the step size chosen by line search. L-BFGS converges in far fewer iterations than vanilla gradient descent for this problem size.

### 3.7 Evaluation Metrics

**Accuracy** — fraction of correctly classified matches:

$$\text{Accuracy} = \frac{1}{n}\sum_{i=1}^{n} \mathbf{1}\{\hat{y}_i = y_i\}$$

**Precision, Recall, F1** for class $k$:

$$\text{Precision}_k = \frac{TP_k}{TP_k + FP_k}, \qquad \text{Recall}_k = \frac{TP_k}{TP_k + FN_k}$$

$$F1_k = 2 \cdot \frac{\text{Precision}_k \cdot \text{Recall}_k}{\text{Precision}_k + \text{Recall}_k}$$

**Macro F1** — unweighted average across all three classes:

$$F1_{\text{macro}} = \frac{1}{K} \sum_{k=0}^{K-1} F1_k$$

We report macro F1 (not weighted) because Draws are a minority class and we care about all three outcomes equally.

### 3.8 Naive Baseline

A useful sanity check is the **majority class baseline**: always predict Home Win (the most common outcome). If our model does not beat this, it has learned nothing useful.

$$\text{Baseline Accuracy} = \frac{\text{\# Home Wins}}{n_{\text{test}}}$$